# Conditional SE(3) Diffusion Training (STAR-MD)

Trains a **conditional** STAR-MD score network on a single protein MD
trajectory using the SinFusion single-trajectory protocol with
Diffusion Forcing.

**What you get:** A model that autoregressively generates MD trajectories
frame-by-frame, conditioned on previous frames.

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive (persistent storage) |
| 2 | Clone repo & init submodules |
| 3 | Configure protein & training |
| 4 | Download & preprocess trajectory |
| 5 | Train (STAR-MD with Diffusion Forcing) |
| 6 | Generate trajectory (autoregressive rollout) |

## Step 1: Environment Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print('Drive mounted')

## Step 2: Clone Repository & Init Submodules

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/JiwonJJeong/winter-gen-pproject.git'

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run(['git', 'submodule', 'update', '--init', '--recursive'], check=True)
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print('data/ -> /content/drive/MyDrive/protein_data/data')
    print('Code ready')
else:
    assert os.path.isdir('gen_model'), 'Run from repo root'
    print(f'Local repo: {os.path.abspath(".")}')

## Step 3: Configuration

Edit the variables below for your protein and training preferences.

In [ ]:
# ---- Protein (single-trajectory, SinFusion-style) ----
PROTEIN   = '4o66_C'   # Protein name (as in atlas.csv)
REPLICA   = '1'        # Which replica (1, 2, or 3)

# ---- Generalization test (ill-posed but informative) ----
# Train on PROTEIN, evaluate on both PROTEIN and EVAL_PROTEIN
# to quantify single-trajectory overfitting vs. generalization.
EVAL_PROTEIN = '6ono_C'  # 76-residue protein from MDGen/STAR-MD test set (never in any training split)

# ---- Training ----
MAX_STEPS  = 200_000   # Total training steps
BATCH_SIZE = 1         # SinFusion default for single-trajectory
LR         = 1e-4
GRAD_CLIP  = 1.0       # Gradient clipping (MDGen default)
EMA_DECAY  = 0.999     # EMA decay (0 = disabled)

# ---- STAR-MD / Diffusion Forcing ----
NUM_FRAMES = 8        # Window length L (paper: 80; 16 for limited data)
MIN_T      = 0.01      # Min diffusion time
MAX_T      = 0.1       # Max diffusion time (narrow noise for Diffusion Forcing)
NS_PER_FRAME = 0.1     # Physical time between stored frames (ns)
CURRICULUM = True       # SinFusion-style curriculum for delta_t

# ---- Paths ----
DATA_DIR = './data'
SAVE_DIR = f'checkpoints/conditional/{PROTEIN}'

print(f'Train protein : {PROTEIN}_R{REPLICA}')
print(f'Eval protein  : {EVAL_PROTEIN}')
print(f'Steps         : {MAX_STEPS:,}')
print(f'Window (L)    : {NUM_FRAMES}')
print(f'Curriculum    : {CURRICULUM}')
print(f'Save to       : {SAVE_DIR}')


## Step 4: Download & Prepare Data

In [ ]:
import os

if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError('data/ not linked to Drive. Run Step 1 first.')

# Download training protein
!python scripts/download_and_prep.py {PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

# Download evaluation protein (for generalization test)
!python scripts/download_and_prep.py {EVAL_PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

for prot in [PROTEIN, EVAL_PROTEIN]:
    traj_path = f'{DATA_DIR}/{prot}/{prot}_R1_latent.npy'
    if os.path.exists(traj_path):
        import numpy as np
        arr = np.load(traj_path)
        print(f'  {prot}: {traj_path}  shape={arr.shape}')
    else:
        print(f'  ERROR: {traj_path} not found')


## Step 5: Train (STAR-MD + Diffusion Forcing)

Runs `gen_model/train_conditional.py` with:
- STAR-MD spatio-temporal attention (block-causal, 2D-RoPE, AdaLN)
- Diffusion Forcing: all L frames noised at same tau, loss at all positions
- SinFusion curriculum for delta_t (gradually increasing temporal range)
- EMA, gradient clipping, cosine LR, mixed precision

In [ ]:
curriculum_flag = '--curriculum' if CURRICULUM else '--no_curriculum'

!python gen_model/train_conditional.py \
    --protein {PROTEIN} \
    --replica {REPLICA} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --max_steps {MAX_STEPS} \
    --lr {LR} \
    --grad_clip {GRAD_CLIP} \
    --ema_decay {EMA_DECAY} \
    --num_frames {NUM_FRAMES} \
    --ns_per_stored_frame {NS_PER_FRAME} \
    --min_t {MIN_T} \
    --max_t {MAX_T} \
    {curriculum_flag} \
    --save_dir {SAVE_DIR} \
    --num_workers 2

## Step 6: Generate Trajectory (Training Protein)

In [ ]:
import glob

TOTAL_FRAMES = 250
DELTA_T      = 0.1
N_STEPS      = 200

ckpts = sorted(glob.glob(f'{SAVE_DIR}/cond-*.ckpt'))
best_ckpt = ckpts[-1] if ckpts else f'{SAVE_DIR}/last.ckpt'
print(f'Checkpoint: {best_ckpt}')

!python gen_model/inference_conditional.py \
    --checkpoint {best_ckpt} \
    --data_dir {DATA_DIR} \
    --protein {PROTEIN} \
    --total_frames {TOTAL_FRAMES} \
    --num_frames {NUM_FRAMES} \
    --delta_t {DELTA_T} \
    --n_steps {N_STEPS} \
    --output outputs/conditional/{PROTEIN}/traj.pt


## Step 7: Evaluate (Training Protein)

Compare generated trajectory against reference MD (all 3 replicas).

In [ ]:
!python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{PROTEIN}/{PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R3_latent.npy \
    --gen_traj outputs/conditional/{PROTEIN}/traj.pt \
    --protein {PROTEIN} --mode conditional --ref_mode all \
    --out_dir outputs/eval/conditional/{PROTEIN}


## Step 8: Visualize (Training Protein)

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

traj_out = f'outputs/conditional/{PROTEIN}/traj.pt'
if os.path.exists(traj_out):
    traj = torch.load(traj_out, map_location='cpu')
    T, N, _ = traj.shape
    ca = traj[:, :, 4:7].numpy()
    rmsd = np.sqrt(((ca - ca[0:1]) ** 2).sum(-1).mean(-1))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(rmsd); ax1.set_xlabel('Frame'); ax1.set_ylabel('RMSD')
    ax1.set_title('Drift from frame 0')
    ax2.plot(ca[0,:,0], ca[0,:,1], '-', lw=0.5, label='Frame 0')
    ax2.plot(ca[-1,:,0], ca[-1,:,1], '-', lw=0.5, label=f'Frame {T-1}')
    ax2.set_aspect('equal'); ax2.legend(); ax2.axis('off')
    ax2.set_title('First vs last (XY)')
    plt.suptitle(f'{PROTEIN} (training protein) — {T} frames')
    plt.tight_layout(); plt.show()
else:
    print('No trajectory found.')


## Step 9: Ablation — Local Spatial Attention

Re-train with `--spatial_sigma 15.0` (SinFusion-inspired local receptive
field in ST attention) and compare against the global attention model
from Step 7.

The Gaussian bias `-(d/sigma)^2` on CA-CA distance means:
- <15A: full attention
- ~30A: strongly attenuated
- >45A: nearly zero

**Hypothesis:** Local attention should reduce overfitting on the single
trajectory by preventing memorization of global pairwise relationships.

In [ ]:
SPATIAL_SIGMA = 15.0
SAVE_DIR_LOCAL = f'checkpoints/conditional_local/{PROTEIN}'

curriculum_flag = '--curriculum' if CURRICULUM else '--no_curriculum'

!python gen_model/train_conditional.py \
    --protein {PROTEIN} \
    --replica {REPLICA} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --max_steps {MAX_STEPS} \
    --lr {LR} \
    --grad_clip {GRAD_CLIP} \
    --ema_decay {EMA_DECAY} \
    --num_frames {NUM_FRAMES} \
    --ns_per_stored_frame {NS_PER_FRAME} \
    --min_t {MIN_T} \
    --max_t {MAX_T} \
    {curriculum_flag} \
    --spatial_sigma {SPATIAL_SIGMA} \
    --save_dir {SAVE_DIR_LOCAL} \
    --num_workers 2


In [ ]:
import glob

# Generate with local-attention model
ckpts_local = sorted(glob.glob(f'{SAVE_DIR_LOCAL}/cond-*.ckpt'))
best_ckpt_local = ckpts_local[-1] if ckpts_local else f'{SAVE_DIR_LOCAL}/last.ckpt'
print(f'Local attention checkpoint: {best_ckpt_local}')

!python gen_model/inference_conditional.py \
    --checkpoint {best_ckpt_local} \
    --data_dir {DATA_DIR} \
    --protein {PROTEIN} \
    --total_frames {TOTAL_FRAMES} \
    --num_frames {NUM_FRAMES} \
    --delta_t {DELTA_T} \
    --n_steps {N_STEPS} \
    --st_num_heads 8 \
    --output outputs/conditional_local/{PROTEIN}/traj.pt

# Evaluate
!python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{PROTEIN}/{PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R3_latent.npy \
    --gen_traj outputs/conditional_local/{PROTEIN}/traj.pt \
    --protein {PROTEIN} --mode conditional --ref_mode all \
    --out_dir outputs/eval/conditional_local/{PROTEIN}


In [ ]:
# Side-by-side comparison: global vs local attention
import pickle, numpy as np

results = {}
for label, eval_dir in [
    ('Global (sigma=0)', f'outputs/eval/conditional/{PROTEIN}'),
    (f'Local (sigma={SPATIAL_SIGMA})', f'outputs/eval/conditional_local/{PROTEIN}'),
]:
    pkl = f'{eval_dir}/gen/eval_results.pkl'
    if os.path.exists(pkl):
        with open(pkl, 'rb') as f:
            data = pickle.load(f)
        if PROTEIN in data and 'JSD' in data[PROTEIN]:
            results[label] = data[PROTEIN]['JSD']

if len(results) == 2:
    print(f'{"Metric":<45s}  {"Global":>8s}  {"Local":>8s}  {"Delta":>8s}')
    print('=' * 75)
    labels = list(results.keys())
    for key in sorted(results[labels[0]].keys()):
        g = results[labels[0]][key]
        l = results[labels[1]][key]
        d = l - g
        better = '<' if d < 0 else '>'
        print(f'{key:<45s}  {g:8.4f}  {l:8.4f}  {d:+8.4f} {better}')
else:
    print('Run both global (Step 7) and local (Step 9) evaluations first.')


In [ ]:
# ── Select best spatial_sigma for all subsequent steps ──────────────
# Auto-select based on mean torsion JSD, or override manually below.

import pickle, numpy as np

AUTO_SELECT = True  # Set False to manually choose BEST_SPATIAL_SIGMA below

if AUTO_SELECT and len(results) == 2:
    labels = list(results.keys())
    # Mean JSD across all features (lower is better)
    mean_jsd = {label: np.mean(list(jsd.values())) for label, jsd in results.items()}
    best_label = min(mean_jsd, key=mean_jsd.get)
    BEST_SPATIAL_SIGMA = SPATIAL_SIGMA if 'Local' in best_label else 0.0
    print(f'Auto-selected: spatial_sigma = {BEST_SPATIAL_SIGMA}')
    print(f'  Global mean JSD: {mean_jsd[labels[0]]:.4f}')
    print(f'  Local  mean JSD: {mean_jsd[labels[1]]:.4f}')
    print(f'  Winner: {best_label}')
else:
    # ---- MANUAL OVERRIDE: set your preferred value here ----
    BEST_SPATIAL_SIGMA = 0.0  # <-- edit this (0.0 = global, 15.0 = local)
    print(f'Manual selection: spatial_sigma = {BEST_SPATIAL_SIGMA}')

# Update checkpoint and output paths for subsequent steps
if BEST_SPATIAL_SIGMA > 0:
    best_ckpt = best_ckpt_local
    BEST_LABEL = f'local (sigma={BEST_SPATIAL_SIGMA})'
else:
    # best_ckpt already set in Step 6
    BEST_LABEL = 'global (sigma=0)'

print(f'\nUsing {BEST_LABEL} model for Steps 10-12.')
print(f'Checkpoint: {best_ckpt}')


## Step 10: Generalization Test

Generate, evaluate, and visualize on `EVAL_PROTEIN` — a held-out protein
from the MDGen/STAR-MD test set (never in any training split).
Quantifies single-trajectory overfitting vs. generalization.

In [ ]:
print(f'Generalization test: {PROTEIN} (train) -> {EVAL_PROTEIN} (held-out)')
print(f'  {EVAL_PROTEIN} is from the MDGen/STAR-MD test set')

# Generate trajectory on held-out protein
!python gen_model/inference_conditional.py \
    --checkpoint {best_ckpt} \
    --data_dir {DATA_DIR} \
    --protein {EVAL_PROTEIN} \
    --total_frames {TOTAL_FRAMES} \
    --num_frames {NUM_FRAMES} \
    --delta_t {DELTA_T} \
    --n_steps {N_STEPS} \
    --output outputs/conditional/{EVAL_PROTEIN}/traj.pt

# Evaluate against all 3 replicas
!python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R3_latent.npy \
    --gen_traj outputs/conditional/{EVAL_PROTEIN}/traj.pt \
    --protein {EVAL_PROTEIN} --mode conditional --ref_mode all \
    --out_dir outputs/eval/conditional/{EVAL_PROTEIN}


In [ ]:
# Visualize generalization trajectory
import torch, numpy as np, matplotlib.pyplot as plt

traj_out = f'outputs/conditional/{EVAL_PROTEIN}/traj.pt'
if os.path.exists(traj_out):
    traj = torch.load(traj_out, map_location='cpu')
    T, N, _ = traj.shape
    ca = traj[:, :, 4:7].numpy()
    rmsd = np.sqrt(((ca - ca[0:1]) ** 2).sum(-1).mean(-1))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(rmsd); ax1.set_xlabel('Frame'); ax1.set_ylabel('RMSD')
    ax1.set_title('Drift from frame 0')
    ax2.plot(ca[0,:,0], ca[0,:,1], '-', lw=0.5, label='Frame 0')
    ax2.plot(ca[-1,:,0], ca[-1,:,1], '-', lw=0.5, label=f'Frame {T-1}')
    ax2.set_aspect('equal'); ax2.legend(); ax2.axis('off')
    ax2.set_title('First vs last (XY)')
    plt.suptitle(f'{EVAL_PROTEIN} (held-out protein) — {T} frames')
    plt.tight_layout(); plt.show()
else:
    print('No trajectory found.')


## Step 11: MDGen Baseline

Download the pre-trained MDGen ATLAS checkpoint from HuggingFace and run
inference on both proteins. MDGen is trained on 1266 ATLAS proteins
(multi-trajectory), so this provides an **upper-bound baseline** for
comparison against our single-trajectory model.

Model weights: https://huggingface.co/bjing-mit/mdgen

In [ ]:
import os, subprocess

# Download MDGen ATLAS checkpoint from HuggingFace
MDGEN_CKPT_DIR = 'checkpoints/mdgen'
MDGEN_CKPT = os.path.join(MDGEN_CKPT_DIR, 'atlas.ckpt')

if not os.path.exists(MDGEN_CKPT):
    os.makedirs(MDGEN_CKPT_DIR, exist_ok=True)
    print('Downloading MDGen ATLAS checkpoint from HuggingFace ...')
    subprocess.run([
        'huggingface-cli', 'download', 'bjing-mit/mdgen', 'atlas.ckpt',
        '--local-dir', MDGEN_CKPT_DIR,
    ], check=True)
    print(f'Downloaded: {MDGEN_CKPT}')
else:
    print(f'MDGen checkpoint exists: {MDGEN_CKPT}')


In [ ]:
import sys, os

mdgen_script = 'extern/mdgen/sim_inference.py'
mdgen_out = 'outputs/mdgen_baseline'
os.makedirs(mdgen_out, exist_ok=True)

# Run MDGen on training protein
print(f'Running MDGen on {PROTEIN} ...')
!python {mdgen_script} \
    --sim_ckpt {MDGEN_CKPT} \
    --data_dir {DATA_DIR} \
    --split gen_model/splits/atlas.csv \
    --pdb_id {PROTEIN} \
    --num_frames 250 --num_rollouts 1 \
    --suffix _R1_latent \
    --xtc \
    --out_dir {mdgen_out}

# Run MDGen on eval protein
print(f'\nRunning MDGen on {EVAL_PROTEIN} ...')
!python {mdgen_script} \
    --sim_ckpt {MDGEN_CKPT} \
    --data_dir {DATA_DIR} \
    --split gen_model/splits/atlas.csv \
    --pdb_id {EVAL_PROTEIN} \
    --num_frames 250 --num_rollouts 1 \
    --suffix _R1_latent \
    --xtc \
    --out_dir {mdgen_out}


In [ ]:
# Evaluate MDGen baseline on both proteins
for prot in [PROTEIN, EVAL_PROTEIN]:
    mdgen_pdb = f'{mdgen_out}/{prot}.pdb'
    if os.path.exists(mdgen_pdb):
        print(f'\n{"="*60}')
        print(f'MDGen baseline evaluation: {prot}')
        print(f'{"="*60}')
        ref_parent = f'outputs/eval/mdgen_baseline/{prot}/ref/{prot}'
        os.makedirs(ref_parent, exist_ok=True)
        !python -c "
import numpy as np, sys; sys.path.insert(0,'extern/mdgen')
from mdgen.utils import atom14_to_pdb
from mdgen.residue_constants import restype_order
import pandas as pd, mdtraj
seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres = seq_df.loc['{prot}', 'seqres']
aatype = np.array([restype_order[c] for c in seqres])
refs = [np.load(f'{DATA_DIR}/{prot}/{prot}_R{{r}}_latent.npy').astype(np.float32) for r in [1,2,3]]
ref = np.concatenate(refs)
atom14_to_pdb(ref, aatype, '{ref_parent}/{prot}.pdb')
t = mdtraj.load('{ref_parent}/{prot}.pdb'); t.superpose(t)
t[0].save('{ref_parent}/{prot}.pdb'); t.save('{ref_parent}/{prot}.xtc')
"
        !python extern/mdgen/scripts/analyze_peptide_sim.py \
            --mddir outputs/eval/mdgen_baseline/{prot}/ref \
            --pdbdir {mdgen_out} \
            --pdb_id {prot} \
            --save --save_name mdgen_eval.pkl \
            --no_msm --plot
    else:
        print(f'MDGen output not found for {prot}')


## Step 12: Naive Single-Trajectory MDGen Baseline

Train MDGen's own architecture on the **same single trajectory** as our
model (no SinFusion anti-overfitting, no STAR-MD attention). This isolates
the contribution of our architectural changes vs. naive single-trajectory
training with MDGen's flow matching approach.

Comparison:
- **Our model** (Steps 6-8): STAR-MD + SinFusion protocol
- **MDGen pre-trained** (Step 10): multi-trajectory upper bound
- **MDGen naive** (this step): same data, MDGen architecture, no anti-overfitting

In [ ]:
import os, subprocess

# Create a single-protein split CSV for MDGen training
import pandas as pd

atlas_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
single_train = atlas_df.loc[[PROTEIN]]
single_val = atlas_df.loc[[PROTEIN]]  # overfit check: val = train

os.makedirs('outputs/mdgen_naive', exist_ok=True)
train_split = f'outputs/mdgen_naive/{PROTEIN}_train.csv'
val_split   = f'outputs/mdgen_naive/{PROTEIN}_val.csv'
single_train.to_csv(train_split)
single_val.to_csv(val_split)
print(f'Created single-protein splits: {train_split}')


In [ ]:
import os

MDGEN_NAIVE_DIR = f'checkpoints/mdgen_naive/{PROTEIN}'
os.makedirs(MDGEN_NAIVE_DIR, exist_ok=True)
os.environ['MODEL_DIR'] = MDGEN_NAIVE_DIR

# Train MDGen on single protein with its ATLAS config
# (num_frames=250, batch_size=1, prepend_ipa, crop up to 256 residues)
# Reduced epochs since we're training on ~1/1000th of the data
MDGEN_EPOCHS = 500

!python extern/mdgen/train.py \
    --sim_condition \
    --train_split {train_split} \
    --val_split {val_split} \
    --data_dir {DATA_DIR} \
    --num_frames 250 \
    --batch_size 1 \
    --prepend_ipa \
    --crop 256 \
    --val_repeat 1 \
    --epochs {MDGEN_EPOCHS} \
    --atlas \
    --ckpt_freq 50 \
    --suffix _R{REPLICA}_latent \
    --no_validate \
    --run_name mdgen_naive_{PROTEIN}


In [ ]:
import glob

# Find latest checkpoint
naive_ckpts = sorted(glob.glob(f'{MDGEN_NAIVE_DIR}/*.ckpt'))
naive_ckpt = naive_ckpts[-1] if naive_ckpts else None

if naive_ckpt:
    print(f'Naive MDGen checkpoint: {naive_ckpt}')
    mdgen_naive_out = 'outputs/mdgen_naive/inference'
    os.makedirs(mdgen_naive_out, exist_ok=True)

    # Run on training protein
    !python extern/mdgen/sim_inference.py \
        --sim_ckpt {naive_ckpt} \
        --data_dir {DATA_DIR} \
        --split gen_model/splits/atlas.csv \
        --pdb_id {PROTEIN} \
        --num_frames 250 --num_rollouts 1 \
        --suffix _R1_latent \
        --xtc \
        --out_dir {mdgen_naive_out}

    # Run on eval protein
    !python extern/mdgen/sim_inference.py \
        --sim_ckpt {naive_ckpt} \
        --data_dir {DATA_DIR} \
        --split gen_model/splits/atlas.csv \
        --pdb_id {EVAL_PROTEIN} \
        --num_frames 250 --num_rollouts 1 \
        --suffix _R1_latent \
        --xtc \
        --out_dir {mdgen_naive_out}

    # Evaluate
    for prot in [PROTEIN, EVAL_PROTEIN]:
        pdb = f'{mdgen_naive_out}/{prot}.pdb'
        if os.path.exists(pdb):
            print(f'\nEvaluating naive MDGen on {prot} ...')
            ref_dir = f'outputs/eval/mdgen_naive/{prot}/ref/{prot}'
            os.makedirs(ref_dir, exist_ok=True)
            !python -c "
import numpy as np, sys; sys.path.insert(0,'extern/mdgen')
from mdgen.utils import atom14_to_pdb
from mdgen.residue_constants import restype_order
import pandas as pd, mdtraj
seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres = seq_df.loc['{prot}', 'seqres']
aatype = np.array([restype_order[c] for c in seqres])
refs = [np.load(f'{DATA_DIR}/{prot}/{prot}_R{{r}}_latent.npy').astype(np.float32) for r in [1,2,3]]
ref = np.concatenate(refs)
atom14_to_pdb(ref, aatype, '{ref_dir}/{prot}.pdb')
t = mdtraj.load('{ref_dir}/{prot}.pdb'); t.superpose(t)
t[0].save('{ref_dir}/{prot}.pdb'); t.save('{ref_dir}/{prot}.xtc')
"
            !python extern/mdgen/scripts/analyze_peptide_sim.py \
                --mddir outputs/eval/mdgen_naive/{prot}/ref \
                --pdbdir {mdgen_naive_out} \
                --pdb_id {prot} \
                --save --save_name mdgen_naive_eval.pkl \
                --no_msm --plot
else:
    print('No naive MDGen checkpoint found. Training may have failed.')
